# 05 · Text Preprocessing

**Goal.** Clean the raw text and make the **Hinglish** (Hindi written in English letters) easier for the
model to read.

**Why it matters.** Social-media text is messy — emojis, links, negations, and romanised Hindi. Tidying
it up and transliterating Hinglish into Devanagari helps the language model understand it.

**What happens below.** Normalise characters, handle emojis and links, tag negations, and transliterate
known Hinglish words into Hindi script.

**Output:** `data/stage2_semantic/` (cleaned, model-ready text).

## 0. Environment & dependencies

Core (`numpy`, `pandas`) always load. No model is loaded in notebook 05 (preprocessing is deterministic and
text-only); MuRIL embedding is notebook 06. Same defensive-flag pattern as Parts 1.1–1.4.

In [1]:
import os, re, sys, json, unicodedata, logging, warnings
from pathlib import Path
from collections import Counter, defaultdict

# Windows consoles default to cp1252 and crash when printing Devanagari; force utf-8 stdout.
for _stream in ("stdout", "stderr"):
    try:
        getattr(sys, _stream).reconfigure(encoding="utf-8")
    except Exception:
        pass

import numpy as np
import pandas as pd

def _try(name):
    try:
        __import__(name); return True
    except Exception:
        return False

HAS_PYARROW = _try("pyarrow") or _try("fastparquet")
PART_VERSION = "1.0"

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("preproc")
log.info("deps -> parquet_engine=%s", HAS_PYARROW)

2026-08-07 21:26:44,425 | INFO | deps -> parquet_engine=True


## 1. Configuration & paths

In [2]:
def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "stage1_augmented").exists() or (d / ".env").exists():
            return d
    return here

PROJECT_ROOT = find_project_root()
AUG_DIR  = PROJECT_ROOT / "data" / "stage1_augmented"
TEMP_DIR = PROJECT_ROOT / "data" / "stage1_temporal"
OUTPUT_DIR = PROJECT_ROOT / "data" / "stage2_semantic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_file(name):
    cands = [Path.cwd() / name, PROJECT_ROOT / "Objective 1" / name]
    for c in cands:
        if c.exists():
            return c
    hits = list(PROJECT_ROOT.glob("**/" + name))
    return hits[0] if hits else None

TRANSLIT_PATH = find_file("translit_map_hi_v1.json")

# negation scope: tag up to N content tokens after a negation cue
NEG_WINDOW = 3
NEG_CUES = {
    "not", "no", "never", "none", "cannot", "cant", "can't", "dont", "don't", "didnt", "didn't",
    "wont", "won't", "isnt", "isn't", "without", "nahi", "nahin", "nhi", "na", "mat", "bina", "naa",
}
NEG_STOP = set(".,!?;:")  # negation scope ends at clause punctuation

log.info("PROJECT_ROOT=%s", PROJECT_ROOT)
log.info("TRANSLIT_MAP=%s", TRANSLIT_PATH)

2026-08-07 21:26:44,972 | INFO | PROJECT_ROOT=hinglish-mental-health-detection


2026-08-07 21:26:44,974 | INFO | TRANSLIT_MAP=hinglish-mental-health-detection\resources\translit_map_hi_v1.json


## 2. Load Objective 1 corpus + the temporal mask

JSONL is the source of truth. We read the augmented TRAIN (1,971 rows) and frozen TEST (2,768 rows), and join
the notebook 04 `temporal_mask` on `record_id` so the preprocessed corpus is self-contained for Objective 3.

In [3]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl(AUG_DIR / "train_augmented.jsonl")
test_df  = load_jsonl(AUG_DIR / "test_frozen.jsonl")

def load_mask(path):
    if not path.exists():
        return {}
    m = {}
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                r = json.loads(line)
                m[r.get("record_id")] = r.get("temporal_mask")
    return m

mask_map = {}
mask_map.update(load_mask(TEMP_DIR / "temporal_features_train.jsonl"))
mask_map.update(load_mask(TEMP_DIR / "temporal_features_test.jsonl"))

for df in (train_df, test_df):
    df["temporal_mask"] = df["record_id"].map(mask_map)
    # belt-and-suspenders: synthetic -> 0, else default 1 if join missing
    df["temporal_mask"] = df.apply(
        lambda r: 0 if bool(r.get("is_synthetic")) else (r["temporal_mask"] if pd.notna(r["temporal_mask"]) else 1),
        axis=1).astype(int)

log.info("loaded train=%d test=%d | temporal_mask joined (train masked-out=%d, test masked-out=%d)",
         len(train_df), len(test_df),
         int((train_df['temporal_mask'] == 0).sum()), int((test_df['temporal_mask'] == 0).sum()))

2026-08-07 21:26:45,395 | INFO | loaded train=1971 test=2768 | temporal_mask joined (train masked-out=125, test masked-out=0)


## 3. Preprocessing components (deterministic, P1)

NFC + Indic cleanup → structured placeholders (`<URL>`/`<USER>`/`<HASHTAG>`) → emoji → sentiment symbol →
**negation-scope tagging** (`happy_NEG`) for `text_negation` → **transliteration** (phrases longest-first,
then words) for `text_muril`. Named entities/English are preserved (only mapped tokens change).

In [4]:
TRANSLIT = json.loads(Path(TRANSLIT_PATH).read_text(encoding="utf-8"))
TR_PHRASES = sorted(TRANSLIT["phrases"].items(), key=lambda kv: -len(kv[0].split()))  # longest-first
TR_WORDS   = TRANSLIT["words"]

URL_RE     = re.compile(r"https?://\S+|www\.\S+")
USER_RE    = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\w+")
WS_RE      = re.compile(r"\s+")

# emoji -> sentiment symbol (small, auditable map; covers the common distress/affect set)
EMOJI_SENT = {}
for ch in "😭😢😞😔😟😥😪😓💔🥀😩😫": EMOJI_SENT[ch] = "<EMO_SAD>"
for ch in "😡😠🤬😤":              EMOJI_SENT[ch] = "<EMO_ANGRY>"
for ch in "😨😰😱😬":              EMOJI_SENT[ch] = "<EMO_FEAR>"
for ch in "😀😃😄😁😊🙂😍🥰😂🤣❤️": EMOJI_SENT[ch] = "<EMO_HAPPY>"

def nfc_clean(t):
    t = unicodedata.normalize("NFC", str(t or ""))
    t = t.replace("‍", "").replace("️", "")  # ZWJ / variation selector
    return WS_RE.sub(" ", t).strip()

def apply_placeholders(t):
    t = URL_RE.sub(" <URL> ", t)
    t = USER_RE.sub(" <USER> ", t)
    t = HASHTAG_RE.sub(" <HASHTAG> ", t)
    return WS_RE.sub(" ", t).strip()

def map_emoji(t):
    out, n = [], 0
    for ch in t:
        if ch in EMOJI_SENT:
            out.append(" " + EMOJI_SENT[ch] + " "); n += 1
        else:
            out.append(ch)
    return WS_RE.sub(" ", "".join(out)).strip(), n

def negation_tag(t):
    toks = t.split()
    out, scope = [], 0
    for w in toks:
        bare = w.lower().strip("".join(NEG_STOP))
        if scope > 0 and bare and not w.startswith("<") and bare not in NEG_CUES:
            out.append(w + "_NEG"); scope -= 1
        else:
            out.append(w)
        if bare in NEG_CUES:
            scope = NEG_WINDOW
        if any(p in w for p in NEG_STOP):
            scope = 0   # clause boundary ends negation scope
    return " ".join(out)

def transliterate(t):
    # Case-preserving: match the map case-insensitively but keep original case on UNMAPPED tokens
    # (the backbone is muril-base-cased, so English NE casing is worth preserving). Devanagari is caseless.
    hits = 0
    s = t
    # phrase-level first (longest-first), case-insensitive
    for ph, dev in TR_PHRASES:
        pat = re.compile(r"\b" + re.escape(ph) + r"\b", re.IGNORECASE)
        s, n = pat.subn(dev, s)
        hits += n
    # token-level
    out = []
    for tok in s.split():
        m = re.match(r"^([^\w<]*)([\w']+)([^\w>]*)$", tok)
        if m and m.group(2).lower() in TR_WORDS:
            out.append(m.group(1) + TR_WORDS[m.group(2).lower()] + m.group(3)); hits += 1
        else:
            out.append(tok)
    return WS_RE.sub(" ", " ".join(out)).strip(), hits

log.info("translit map v%s: %d phrases, %d words; emoji map: %d entries",
         TRANSLIT["version"], len(TR_PHRASES), len(TR_WORDS), len(EMOJI_SENT))

2026-08-07 21:26:45,415 | INFO | translit map v1.0: 35 phrases, 95 words; emoji map: 32 entries


## 4. The `ruminative_repetition` flag (deferred from notebook 01 D4 / 1.4 D5)

A documented lexical proxy for the paper's ruminative-repetition signal: flag a post if (a) any content
token repeats ≥ 3 times, or (b) two of its sentences are near-identical (token-Jaccard ≥ 0.8). It needs
sentence segmentation, which is why it waited for this preprocessing stage. Computed on the cleaned
romanized text (pre-transliteration) so token identity is stable.

In [5]:
SENT_SPLIT = re.compile(r"[.!?\n]+")

# function words excluded from "content token repetition" so the flag isn't tripped by hai/nahi/the/...
RUMI_STOP = {
    "hai", "hain", "nahi", "nahin", "nhi", "tha", "thi", "kya", "kyu", "kyun", "mera", "meri", "mere",
    "mujhe", "muje", "tum", "kuch", "bahut", "bhai", "raha", "rahi", "rahe", "kar", "karo", "hona",
    "and", "the", "for", "that", "this", "with", "but", "you", "are", "was", "have", "has", "not",
    "main", "hoon", "hota", "hoti", "aur", "bhi", "wala", "wali", "sab", "log", "yaar", "abhi",
}

def ruminative_flag(t):
    # (a) a meaningful content token (len>=4, not a function word) repeated >= 3 times
    toks = [w for w in re.findall(r"[\w']+", t.lower()) if len(w) >= 4 and w not in RUMI_STOP]
    if toks:
        top, cnt = Counter(toks).most_common(1)[0]
        if cnt >= 3:
            return True
    # (b) two near-identical sentences, each with >= 4 content tokens (genuine rumination, not echoes)
    sents = [s.strip() for s in SENT_SPLIT.split(t) if s.strip()]
    sset = [set(w for w in re.findall(r"[\w']+", s.lower()) if len(w) >= 3) for s in sents]
    for i in range(len(sset)):
        for j in range(i + 1, len(sset)):
            a, b = sset[i], sset[j]
            if len(a) >= 4 and len(b) >= 4:
                if len(a & b) / len(a | b) >= 0.8:
                    return True
    return False

log.info("ruminative flag ready (content-token repeat>=3 or near-dup sentences>=4 tokens)")

2026-08-07 21:26:45,428 | INFO | ruminative flag ready (content-token repeat>=3 or near-dup sentences>=4 tokens)


## 5. Run the pipeline over TRAIN + TEST (split kept separate — P3)

In [6]:
def preprocess_frame(df):
    text_muril, text_neg, translit_hits, rumi, emoji_hits = [], [], [], [], []
    base_col = "post_text_clean" if "post_text_clean" in df.columns else "post_text"
    for raw in df[base_col].tolist():
        c = nfc_clean(raw)
        c = apply_placeholders(c)
        c, ne = map_emoji(c)
        neg = negation_tag(c)
        muril, th = transliterate(c)
        text_muril.append(muril); text_neg.append(neg)
        translit_hits.append(th); emoji_hits.append(ne)
        rumi.append(ruminative_flag(c))
    out = df.copy()
    out["text_muril"] = text_muril
    out["text_negation"] = text_neg
    out["translit_map_hits"] = translit_hits
    out["emoji_sub_count"] = emoji_hits
    out["ruminative_repetition"] = rumi
    return out

pre_train = preprocess_frame(train_df)
pre_test  = preprocess_frame(test_df)

# carry-in invariant: row counts preserved, no row crosses the split
assert len(pre_train) == len(train_df) and len(pre_test) == len(test_df), "row count changed!"
assert set(pre_train["platform"].unique()) <= {"youtube", "synthetic"}, "non-train platform leaked into TRAIN!"
assert set(pre_test["platform"].unique()) <= {"reddit", "twitter"}, "non-test platform in frozen TEST!"
log.info("preprocessed: train=%d test=%d", len(pre_train), len(pre_test))
log.info("translit hits/row: train mean=%.2f test mean=%.2f",
         float(np.mean(pre_train['translit_map_hits'])), float(np.mean(pre_test['translit_map_hits'])))

2026-08-07 21:26:47,576 | INFO | preprocessed: train=1971 test=2768


2026-08-07 21:26:47,578 | INFO | translit hits/row: train mean=0.92 test mean=1.38


## 6. Persist + report

In [7]:
def write_jsonl(frame, path):
    with open(path, "w", encoding="utf-8") as fh:
        for _, r in frame.iterrows():
            fh.write(json.dumps(r.to_dict(), ensure_ascii=False, default=str) + "\n")

write_jsonl(pre_train, OUTPUT_DIR / "corpus_preprocessed_train.jsonl")
write_jsonl(pre_test,  OUTPUT_DIR / "corpus_preprocessed_test.jsonl")
for name, frame in [("corpus_preprocessed_train", pre_train), ("corpus_preprocessed_test", pre_test)]:
    try:
        frame.to_parquet(OUTPUT_DIR / (name + ".parquet"), index=False)
        log.info("wrote %s.parquet", name)
    except Exception as e:
        log.warning("parquet %s skipped (%s); jsonl is source of truth", name, e)

def translit_coverage(frame):
    n_any = int((frame["translit_map_hits"] > 0).sum())
    return {"rows_with_translit": n_any, "rows_total": int(len(frame)),
            "pct_rows_with_translit": round(100 * n_any / max(1, len(frame)), 1),
            "mean_hits_per_row": round(float(frame["translit_map_hits"].mean()), 3)}

report = {
    "part": "stage2_part2.1_preprocessing",
    "version": PART_VERSION,
    "principle": "P1 (no human); P2 (text primary); P3 (split kept);  transliteration;  OOV->subword",
    "decisions": {
        "D1_transliteration": "offline mapping table translit_map_hi_v1.json (Google Translate API = scale-up)",
        "D5_ruminative_repetition": "materialised here (lexical proxy)",
    },
    "translit_map_version": TRANSLIT["version"],
    "translit_map_size": {"phrases": len(TR_PHRASES), "words": len(TR_WORDS)},
    "n_train": int(len(pre_train)), "n_test": int(len(pre_test)),
    "translit_coverage_train": translit_coverage(pre_train),
    "translit_coverage_test": translit_coverage(pre_test),
    "emoji_subs_train": int(pre_train["emoji_sub_count"].sum()),
    "emoji_subs_test": int(pre_test["emoji_sub_count"].sum()),
    "ruminative_flag_train": int(pre_train["ruminative_repetition"].sum()),
    "ruminative_flag_test": int(pre_test["ruminative_repetition"].sum()),
    "temporal_mask_passthrough": {
        "train_masked_out": int((pre_train["temporal_mask"] == 0).sum()),
        "test_masked_out": int((pre_test["temporal_mask"] == 0).sum()),
    },
}
(OUTPUT_DIR / "preprocess_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))

2026-08-07 21:26:49,062 | INFO | wrote corpus_preprocessed_train.parquet


2026-08-07 21:26:49,066 | WARNING | parquet corpus_preprocessed_test skipped (("Expected bytes, got a 'int' object", 'Conversion failed for column parent_id with type object')); jsonl is source of truth


{
  "part": "stage2_part2.1_preprocessing",
  "version": "1.0",
  "principle": "P1 (no human); P2 (text primary); P3 (split kept);  transliteration;  OOV->subword",
  "decisions": {
    "D1_transliteration": "offline mapping table translit_map_hi_v1.json (Google Translate API = scale-up)",
    "D5_ruminative_repetition": "materialised here (lexical proxy)"
  },
  "translit_map_version": "1.0",
  "translit_map_size": {
    "phrases": 35,
    "words": 95
  },
  "n_train": 1971,
  "n_test": 2768,
  "translit_coverage_train": {
    "rows_with_translit": 410,
    "rows_total": 1971,
    "pct_rows_with_translit": 20.8,
    "mean_hits_per_row": 0.917
  },
  "translit_coverage_test": {
    "rows_with_translit": 1048,
    "rows_total": 2768,
    "pct_rows_with_translit": 37.9,
    "mean_hits_per_row": 1.384
  },
  "emoji_subs_train": 638,
  "emoji_subs_test": 298,
  "ruminative_flag_train": 180,
  "ruminative_flag_test": 687,
  "temporal_mask_passthrough": {
    "train_masked_out": 125,
    "te

## 7. Transliteration sanity check (anchor terms) + worked samples

In [8]:
ANCHOR_CHECKS = [
    ("akelapan", "अकेलापन"), ("marna", "मरना"), ("rona", "रोना"), ("pareshan", "परेशान"),
    ("khudkushi", "खुदकुशी"), ("jeena nahi chahta", "जीना नहीं चाहता"), ("thak gaya", "थक गया"),
]
print("--- transliteration unit checks ---")
ok = 0
for src, expect in ANCHOR_CHECKS:
    got, _ = transliterate(src)
    status = "OK" if expect in got else "MISS"
    ok += (expect in got)
    print(f"  [{status}] {src!r:32} -> {got!r}")
assert ok == len(ANCHOR_CHECKS), "a transliteration anchor check failed!"

print("\n--- worked samples (real rows with the most transliteration hits) ---")
sample = pre_train.sort_values("translit_map_hits", ascending=False).head(3)
if int(sample["translit_map_hits"].max() or 0) == 0:
    sample = pre_test.sort_values("translit_map_hits", ascending=False).head(3)
for _, r in sample.iterrows():
    print("  RAW   :", str(r.get("post_text_clean"))[:120])
    print("  MURIL :", str(r["text_muril"])[:120])
    print("  NEG   :", str(r["text_negation"])[:120], "\n")

--- transliteration unit checks ---
  [OK] 'akelapan'                       -> 'अकेलापन'
  [OK] 'marna'                          -> 'मरना'
  [OK] 'rona'                           -> 'रोना'
  [OK] 'pareshan'                       -> 'परेशान'
  [OK] 'khudkushi'                      -> 'खुदकुशी'
  [OK] 'jeena nahi chahta'              -> 'जीना नहीं चाहता'
  [OK] 'thak gaya'                      -> 'थक गया'

--- worked samples (real rows with the most transliteration hits) ---
[real post text redacted]
[real post text redacted]
[real post text redacted]

[real post text redacted]
[real post text redacted]
[real post text redacted]

[real post text redacted]
[real post text redacted]
[real post text redacted]



## 8. Preprocessing card & This step is complete when

**notebook 05 is complete when:** `corpus_preprocessed_{train,test}.jsonl` exist with `text_muril`,
`text_negation`, `translit_map_hits`, `ruminative_repetition`; transliteration anchor checks pass;
the split + `temporal_mask` are preserved; coverage/emoji/rumination counts are reported. **No human.**

In [9]:
print("=" * 64); print("Objective 2 · notebook 05 — PREPROCESSING CARD"); print("=" * 64)
print("rows (train / test)        :", len(pre_train), "/", len(pre_test))
print("translit coverage (train)  :", translit_coverage(pre_train)["pct_rows_with_translit"], "% of rows")
print("translit coverage (test)   :", translit_coverage(pre_test)["pct_rows_with_translit"], "% of rows")
print("emoji subs (train/test)    :", int(pre_train["emoji_sub_count"].sum()), "/", int(pre_test["emoji_sub_count"].sum()))
print("ruminative flag (train/test):", int(pre_train["ruminative_repetition"].sum()), "/", int(pre_test["ruminative_repetition"].sum()))
print("temporal_mask passthrough  : train masked-out", int((pre_train["temporal_mask"] == 0).sum()),
      "| test masked-out", int((pre_test["temporal_mask"] == 0).sum()))
dod = all([
    (OUTPUT_DIR / "corpus_preprocessed_train.jsonl").exists(),
    (OUTPUT_DIR / "corpus_preprocessed_test.jsonl").exists(),
    (OUTPUT_DIR / "preprocess_report.json").exists(),
    "text_muril" in pre_train.columns and "text_negation" in pre_train.columns,
    "ruminative_repetition" in pre_train.columns,
    len(pre_train) == len(train_df) and len(pre_test) == len(test_df),
])
print("\nCompletion check:", "PASS" if dod else "INCOMPLETE")

Objective 2 · notebook 05 — PREPROCESSING CARD
rows (train / test)        : 1971 / 2768
translit coverage (train)  : 20.8 % of rows
translit coverage (test)   : 37.9 % of rows
emoji subs (train/test)    : 638 / 298
ruminative flag (train/test): 180 / 687
temporal_mask passthrough  : train masked-out 125 | test masked-out 0

Completion check: PASS
